# Hypothyroidism Detection — Analysis & Model Comparison

The following project report aims to develop an efficient and accurate model to predict the likelihood of a patient having hypothyroid. This is achieved by training classification models on the provided dataset, which contains labeled data indicating the presence or absence of hypothyroidism. The goal is to preprocess the data, analyze it for insights, and apply various classification algorithms to determine the best-performing model based on multiple evaluation metrics.

In [ ]:
# import the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# to access the dataset
df = pd.read_csv("hypothyroid_classification.csv")

## Data Exploration

In [ ]:
# to get information about the dataset
display(df.head())
display(df.describe())
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   age                         3772 non-null   object
 1   sex                         3772 non-null   object
 2   on thyroxine                3772 non-null   object
 3   query on thyroxine          3772 non-null   object
 4   on antithyroid medication   3772 non-null   object
 5   sick                        3772 non-null   object
 6   pregnant                    3772 non-null   object
 7   thyroid surgery             3772 non-null   object
 8   I131 treatment              3772 non-null   object
 9   query hypothyroid           3772 non-null   object
 10  query hyperthyroid          3772 non-null   object
 11  lithium                     3772 non-null   object
 12  goitre                      3772 non-null   object
 13  tumor                       3772 non-null   obje

## Data Processing and Cleaning

In [ ]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

In [ ]:
# Define columns in the dataset
numerical_columns = ['age', 'TSH', 'T3', 'TT4', 'T4U', 'FTI']
categorical_columns = [
    'on thyroxine', 'query on thyroxine', 'on antithyroid medication', 'sick',
    'pregnant', 'thyroid surgery', 'I131 treatment', 'query hypothyroid',
    'query hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych',
    'TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured',
    'TBG measured', 'referral source', 'sex']

In [ ]:
# Convert numerical columns to float, coerce errors to NaN
for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Convert boolean 't'/'f' columns to 1/0
boolean_columns = [
    'on thyroxine', 'query on thyroxine', 'on antithyroid medication', 'sick',
    'pregnant', 'thyroid surgery', 'I131 treatment', 'query hypothyroid',
    'query hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych',
    'TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured',
    'TBG measured']

for col in boolean_columns:
    df[col] = df[col].map({'t': 1, 'f': 0})

In [ ]:
# Encode the target variable
print(f"Unique values in binaryClass before mapping: {df['binaryClass'].unique()}")
df['binaryClass'] = df['binaryClass'].map({'P': 1, 'N': 0})
df['binaryClass'] = df['binaryClass'].astype('Int64')
print(f"Unique values in binaryClass after conversion: {df['binaryClass'].unique()}")

Unique values in binaryClass before mapping: ['P' 'N']
Unique values in binaryClass after conversion: <IntegerArray>
[1, 0]
Length: 2, dtype: Int64

## Missing Value Identification and Treatment

In [ ]:
# Check the missing values
print("THE NUMBER OF MISSING VALUES IN EACH COLUMN:\n")
display(df.isna().sum())

THE NUMBER OF MISSING VALUES IN EACH COLUMN:

age                             1
sex                            150
TSH                            369
T3                              769
TT4                             231
T4U                             387
FTI                             385
TBG                            3772
(remaining boolean/categorical columns: 0 missing)

In [ ]:
# Visualize missing values
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
# 1. Drop columns that are entirely missing (e.g. TBG)
df.dropna(axis=1, how='all', inplace=True)

In [ ]:
# 2. Impute remaining missing values: mean for numerical, mode for categorical
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].mean())

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

display(df.isna().sum())

## Exploratory Data Analysis (EDA)

In [ ]:
# Traffic-free EDA: boxplots for outlier visualization
fig, ax = plt.subplots(nrows=len(numerical_columns), ncols=1, figsize=(10, 20))
for i, col in enumerate(numerical_columns):
    sns.boxplot(x=df[col], ax=ax[i])
    ax[i].set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

**Insights:**
- **Age:** Median around 50.
- **TSH:** Numerous outliers above 100.
- **T3:** Many outliers present.
- **TT4, T4U, FTI:** Many outliers present.

In [ ]:
# Histograms and density plots
fig, ax = plt.subplots(nrows=len(numerical_columns), ncols=2, figsize=(15, 20))
for i, col in enumerate(numerical_columns):
    ax[i, 0].hist(df[col], bins=50)
    ax[i, 0].set_title(f"Histogram of {col}")
    sns.kdeplot(df[col], ax=ax[i, 1])
    ax[i, 1].set_title(f"Density Plot of {col}")
plt.tight_layout()
plt.show()

**Insights:**
- **Age:** Roughly normal, potential outliers at the higher end.
- **TSH:** Highly skewed with extreme outliers.
- **T3:** Roughly normal with potential outliers.
- **TT4:** Slightly skewed with potential outliers.
- **T4U:** Mostly normal but with some outliers.
- **FTI:** Slightly skewed with potential outliers.

In [ ]:
# Correlation heatmap for numerical columns
correlation_matrix = df[numerical_columns].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap for Numerical Columns')
plt.show()

**Insights:**
- Strong positive correlation between TT4 and T4U.
- Moderate positive correlation between T3 and TT4, and between TT4 and FTI.
- Age and TSH show weak correlation with other variables.

## Identification and Treatment of Outliers

In [ ]:
# 1. IQR Method
iqr = {}
for col in numerical_columns:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr[col] = q3 - q1

outlier_counts_iqr = {}
for col, iqr_val in iqr.items():
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    outlier_counts_iqr[col] = (df[col] < (q1 - 1.5*iqr_val)) | (df[col] > (q3 + 1.5*iqr_val))
outlier_counts_iqr = pd.DataFrame(outlier_counts_iqr).sum(axis=0)
print("Outlier counts (IQR) per column:")
print(outlier_counts_iqr)

Outlier counts (IQR) per column:
age      1
TSH    258
T3     456
TT4    215
T4U    205
FTI    277
dtype: int64

In [ ]:
# 2. Z-Score Method
from scipy import stats
z_scores = {col: stats.zscore(df[col]) for col in numerical_columns}
outlier_counts_zscore = {col: (z > 3) | (z < -3) for col, z in z_scores.items()}
outlier_counts_zscore = pd.DataFrame(outlier_counts_zscore).sum(axis=0)
print("Outlier counts (Z-score) per column:")
print(outlier_counts_zscore)

Outlier counts (Z-score) per column:
age     1
TSH    39
T3     54
TT4    54
T4U    74
FTI    79
dtype: int64

In [ ]:
# 3. DBSCAN Method
from sklearn.cluster import DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
outlier_counts_dbscan = {}
for col in numerical_columns:
    dbscan.fit(df[[col]])
    outlier_counts_dbscan[col] = (dbscan.labels_ == -1)
outlier_counts_dbscan = pd.DataFrame(outlier_counts_dbscan).sum(axis=0)
print("Outlier counts (DBSCAN) per column:")
print(outlier_counts_dbscan)

Outlier counts (DBSCAN) per column:
age     74
TSH    156
T3       9
TT4    432
T4U      0
FTI    396
dtype: int64

In [ ]:
# 4. Modified Z-Score Method
threshold = 3.5
modified_z_scores = {}
for col in numerical_columns:
    median = df[col].median()
    mad = np.median(np.abs(df[col] - median))
    modified_z_scores[col] = 0.6745 * (df[col] - median) / mad

outlier_counts_modified_z = {col: (np.abs(mzs) > threshold) for col, mzs in modified_z_scores.items()}
outlier_counts_modified_z = pd.DataFrame(outlier_counts_modified_z).sum(axis=0)
print("Outlier counts (Modified Z-score) per column:")
print(outlier_counts_modified_z)

Outlier counts (Modified Z-score) per column:
age      1
TSH    266
T3     205
TT4    112
T4U    109
FTI    154
dtype: int64

### Conclusions from Outlier Detection

**Age:** Very few outliers by IQR/Z-score/Modified Z-score; DBSCAN detects more, hinting at a cluster of older individuals. Distribution is roughly normal. **Treatment: IQR method.**

**TSH:** High number of outliers across IQR, Modified Z-score, and DBSCAN; distribution is highly skewed with extreme outliers. **Treatment: IQR method** (robust to non-normal distributions).

**T3:** Roughly normal with significant outliers per IQR/Modified Z-score. **Treatment: IQR method.**

**TT4:** High number of outliers by IQR/Modified Z-score; slightly skewed distribution. **Treatment: Z-score method.**

**T4U:** Mostly normal distribution with few outliers. **Treatment: Z-score method.**

**FTI:** Slightly skewed distribution with many outliers detected by DBSCAN/Modified Z-score/IQR. **Treatment: IQR method.**

In [ ]:
def drop_outliers(df, outlier_mask):
    return df[~outlier_mask], df[outlier_mask]

# Treat outliers using IQR for age, TSH, T3, FTI
iqr_columns = ['age', 'TSH', 'T3', 'FTI']
outliers_iqr = pd.Series(False, index=df.index)
for col in iqr_columns:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr_val = q3 - q1
    outliers = (df[col] < (q1 - 1.5*iqr_val)) | (df[col] > (q3 + 1.5*iqr_val))
    outliers_iqr = outliers_iqr | outliers
df_iqr_cleaned, df_iqr_outliers = drop_outliers(df, outliers_iqr)

In [ ]:
# Treat outliers using Z-score for TT4, T4U
z_score_columns = ['TT4', 'T4U']
outliers_zscore = pd.Series(False, index=df.index)
for col in z_score_columns:
    z = stats.zscore(df[col])
    outliers = pd.Series((z > 3) | (z < -3), index=df.index)
    outliers_zscore = outliers_zscore | outliers
df_zscore_cleaned, df_zscore_outliers = drop_outliers(df, outliers_zscore)

# Combine cleaned data
cleaned_data = pd.concat([df_iqr_cleaned, df_zscore_cleaned]).drop_duplicates().reset_index(drop=True)
data = pd.DataFrame(cleaned_data)
display(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3613 entries, 0 to 3612
Data columns (total 29 columns): ...
memory usage: 822.2+ KB

## Feature Engineering

In [ ]:
# One-hot encode categorical variables
data_encoded = pd.get_dummies(data, columns=['sex', 'referral source'])

numerics = data_encoded.select_dtypes(include=['float64']).columns
binary = data_encoded.select_dtypes(include=['int64']).columns
print("Numerical columns:", list(numerics))

Numerical columns: ['age', 'TSH', 'T3', 'TT4', 'T4U', 'FTI']

In [ ]:
# Standard scaling on numerical columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data_scaled = data_encoded.copy()
data_scaled[numerics] = scaler.fit_transform(data_encoded[numerics])

## Multicollinearity

In [ ]:
# Detect highly correlated features (correlation > 0.7)
corr_matrix = data_scaled.corr()
highly_correlated_features = [
    (corr_matrix.columns[i], corr_matrix.columns[j])
    for i in range(len(corr_matrix.columns))
    for j in range(i+1, len(corr_matrix.columns))
    if abs(corr_matrix.iloc[i, j]) > 0.7
]
print("Highly correlated features (correlation > 0.7):")
for pair in highly_correlated_features:
    print(f"{pair[0]} and {pair[1]} are highly correlated")

Highly correlated features (correlation > 0.7):
TT4 and FTI are highly correlated
T4U measured and FTI measured are highly correlated
sex_F and sex_M are highly correlated
referral source_SVI and referral source_other are highly correlated

In [ ]:
# Drop redundant features to reduce multicollinearity
features_to_drop = {pair[1] for pair in highly_correlated_features}
data_reduced = data_scaled.drop(columns=features_to_drop)

In [ ]:
# Correlation with the target variable
target_corr = data_scaled.corr()['binaryClass'].abs()
filtered_features = target_corr[target_corr > 0.1].index
print("Filtered features based on correlation with target:")
print(list(filtered_features))

Filtered features based on correlation with target:
['TSH', 'T3', 'TT4', 'FTI', 'binaryClass']

## Model Building and Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

X = data_reduced.drop('binaryClass', axis=1)
y = data_reduced['binaryClass']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

def evaluate_model_cv(model, X, y, cv_folds=5):
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    accuracies, precisions, recalls, f1s, aucs = [], [], [], [], []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        accuracies.append(accuracy_score(y_test, y_pred))
        precisions.append(precision_score(y_test, y_pred, average='weighted'))
        recalls.append(recall_score(y_test, y_pred, average='weighted'))
        f1s.append(f1_score(y_test, y_pred, average='weighted'))
        aucs.append(roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan)
    return {
        'Mean Accuracy': np.mean(accuracies),
        'Mean Precision': np.mean(precisions),
        'Mean Recall': np.mean(recalls),
        'Mean F1 Score': np.mean(f1s),
        'Mean ROC AUC': np.mean([a for a in aucs if a == a])
    }

### Logistic Regression
A statistical model used for binary classification that estimates the probability of a binary outcome.

In [ ]:
model_lr = LogisticRegression(max_iter=1000)
results_lr = evaluate_model_cv(model_lr, X_scaled, y)
for metric, score in results_lr.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9626
Mean Precision: 0.9618
Mean Recall: 0.9626
Mean F1 Score: 0.9582
Mean ROC AUC: 0.9743

### Decision Tree
A model that splits data into subsets based on feature values, creating a tree-like structure — useful for non-linear relationships.

In [ ]:
model_dt = DecisionTreeClassifier()
results_dt = evaluate_model_cv(model_dt, X_scaled, y)
for metric, score in results_dt.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9947
Mean Precision: 0.9948
Mean Recall: 0.9947
Mean F1 Score: 0.9948
Mean ROC AUC: 0.9846

### Support Vector Machine (SVM)
Finds the optimal hyperplane separating classes — effective in high-dimensional spaces.

In [ ]:
model_svm = SVC(probability=True)
results_svm = evaluate_model_cv(model_svm, X_scaled, y)
for metric, score in results_svm.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9477
Mean Precision: 0.9490
Mean Recall: 0.9477
Mean F1 Score: 0.9358
Mean ROC AUC: 0.9834

### Neural Network
An MLP classifier that learns complex, non-linear patterns.

In [ ]:
model_nn = MLPClassifier(max_iter=1000)
results_nn = evaluate_model_cv(model_nn, X_scaled, y)
for metric, score in results_nn.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9817
Mean Precision: 0.9814
Mean Recall: 0.9817
Mean F1 Score: 0.9813
Mean ROC AUC: 0.9789

### Random Forest
An ensemble of decision trees that reduces overfitting and improves predictive performance.

In [ ]:
model_rf = RandomForestClassifier()
results_rf = evaluate_model_cv(model_rf, X_scaled, y)
for metric, score in results_rf.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9939
Mean Precision: 0.9941
Mean Recall: 0.9939
Mean F1 Score: 0.9940
Mean ROC AUC: 0.9993

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_scaled, y)
best_model = grid_search.best_estimator_
print("Best parameters found by Grid Search:", grid_search.best_params_)

Best parameters found by Grid Search: {'max_depth': 30, 'min_samples_split': 5, 'n_estimators': 50}

In [ ]:
best_results = evaluate_model_cv(best_model, X_scaled, y)
for metric, score in best_results.items():
    print(f"{metric}: {score}")

Mean Accuracy: 0.9945
Mean Precision: 0.9946
Mean Recall: 0.9945
Mean F1 Score: 0.9945
Mean ROC AUC: 0.9994

In [ ]:
# Feature importance from the best model
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), features[indices], rotation=90)
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
cv_results = {
    "Logistic Regression": results_lr,
    "Decision Tree": results_dt,
    "Support Vector Machine": results_svm,
    "Neural Network": results_nn,
    "Random Forest": results_rf
}

print("{:<25} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"))
for name, metrics in cv_results.items():
    print("{:<25} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f}".format(
        name, metrics['Mean Accuracy'], metrics['Mean Precision'],
        metrics['Mean Recall'], metrics['Mean F1 Score'], metrics['Mean ROC AUC']))

best_model_name = max(cv_results, key=lambda x: cv_results[x]['Mean Accuracy'])
print("\nBest performing model:", best_model_name)

Model                     Accuracy     Precision    Recall       F1 Score     ROC AUC     
Logistic Regression       0.9626       0.9618       0.9626       0.9582       0.9743      
Decision Tree             0.9947       0.9948       0.9947       0.9948       0.9846      
Support Vector Machine    0.9477       0.9490       0.9477       0.9358       0.9834      
Neural Network            0.9817       0.9814       0.9817       0.9813       0.9789      
Random Forest             0.9939       0.9941       0.9939       0.9940       0.9993      

Best performing model: Decision Tree

## Conclusion

The model comparison reveals that the **Decision Tree** performs best overall, with the highest accuracy (0.9947), precision (0.9948), recall (0.9947), and F1 score (0.9948). **Random Forest** is close behind and shows the strongest ROC AUC (0.9993), indicating excellent discriminatory power. Neural Network and SVM are still effective but score slightly lower across metrics. TSH, TT4, and FTI emerged as the most predictive features, consistent with their clinical relevance to thyroid function.